# Notebook 7 — Experiments 7 to 10: Features and Tuning
**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

Takes the best baseline model from Experiment 6 and tests three richer feature
configurations, then tunes the winning combination.

| Exp | Feature Set | Fills |
|---|---|---|
| 7 | Character n-grams | Table 2 |
| 8 | TF-IDF + social media features | Table 2 |
| 9 | Hybrid feature set | Table 2 |
| 10 | Hyperparameter-tuned final model | Table 3 |

Only one model is carried forward, which keeps this stage to four training runs
rather than twenty-eight.

Subgroup Macro F1 is reported for every configuration alongside overall accuracy.
If the emoji-heavy and slang-heavy subgroups improve under Experiment 8 but the
overall score barely moves, that is a more interesting finding than the headline
number, and it is direct evidence that explicit feature engineering reduces
linguistic bias.

## Cell 1: Setup

In [2]:
!pip install -q emoji xgboost scikit-learn pandas pyarrow

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")
from thesis_utils import *

import pandas as pd, numpy as np, scipy.sparse as sp, pickle, time, json
from sklearn.model_selection import GridSearchCV

Mounted at /content/drive
thesis_utils loaded.
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal


## Cell 2: Load Data and the Winning Baseline

Reads the best model name recorded by Experiment 6. If Notebook 6 has not been
run, this cell will fail — that is intentional.

In [3]:
D = PATHS["data"]

tw_train = pd.read_parquet(D / "tw_train.parquet")
tw_test  = pd.read_parquet(D / "tw_test.parquet")

y_train = tw_train["sentiment"].values
y_test  = tw_test["sentiment"].values
sg_test = tw_test["subgroup_primary"].values

with open(D / "best_baseline.json") as f:
    best_info = json.load(f)

BEST_MODEL = best_info["best_model"]
print(f"Best baseline from Exp 6 : {BEST_MODEL}")
print(f"Its FS1 macro F1         : {best_info['macro_f1']:.4f}")
print(f"\nTrain rows: {len(tw_train):,}   Test rows: {len(tw_test):,}")

Best baseline from Exp 6 : LogisticRegression
Its FS1 macro F1         : 0.5672

Train rows: 45,615   Test rows: 12,284


## Cell 3: Model Factory

Rebuilds the winning model type with fresh parameters for each feature set.
The model is retrained per configuration because the feature space changes.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

def make_model(name):
    """Return a fresh untrained instance of the named model."""
    if name == "LogisticRegression":
        return LogisticRegression(C=1, max_iter=2000, solver="liblinear",
                                  random_state=SEED)
    if name == "LinearSVM":
        return CalibratedClassifierCV(
            LinearSVC(C=1, max_iter=3000, random_state=SEED, dual="auto"),
            cv=3, method="sigmoid")
    if name == "MultinomialNB":
        return MultinomialNB(alpha=0.5)
    if name == "RandomForest":
        return RandomForestClassifier(n_estimators=200, max_features="sqrt",
                                      random_state=SEED, n_jobs=-1)
    if name == "XGBoost":
        return XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=5,
                             random_state=SEED, n_jobs=-1,
                             eval_metric="mlogloss", tree_method="hist")
    raise ValueError(f"Unknown model: {name}")

NEEDS_ENCODING = BEST_MODEL == "XGBoost"
print(f"Model factory ready. Label encoding required: {NEEDS_ENCODING}")

Model factory ready. Label encoding required: False


## Cell 4: Feature Set Builders

Four builders. Each fits on the training partition only, then transforms the test
partition. The numeric social-media features are scaled with MinMaxScaler because
Multinomial Naive Bayes cannot accept negative values.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

SOCIAL_COLS = ["emoji_density", "slang_ratio", "punct_intensity"]

def fs1_tfidf(train_df, test_df):
    """Baseline: TF-IDF unigram + bigram."""
    vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95,
                          max_features=50_000, sublinear_tf=True)
    return vec.fit_transform(train_df["text_clean"]), vec.transform(test_df["text_clean"]), vec

def fs2_char(train_df, test_df):
    """Exp 7: character n-grams. Catches slang spellings word features miss."""
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2,
                          max_features=30_000, sublinear_tf=True)
    return vec.fit_transform(train_df["text_clean"]), vec.transform(test_df["text_clean"]), vec

def fs3_social(train_df, test_df):
    """Exp 8: TF-IDF plus three engineered linguistic features."""
    Xtr_t, Xte_t, vec = fs1_tfidf(train_df, test_df)
    scaler = MinMaxScaler()
    num_tr = scaler.fit_transform(train_df[SOCIAL_COLS].values)
    num_te = scaler.transform(test_df[SOCIAL_COLS].values)
    Xtr = sp.hstack([Xtr_t, sp.csr_matrix(num_tr)]).tocsr()
    Xte = sp.hstack([Xte_t, sp.csr_matrix(num_te)]).tocsr()
    return Xtr, Xte, (vec, scaler)

def fs4_hybrid(train_df, test_df):
    """Exp 9: word TF-IDF + char n-grams + engineered features."""
    Xtr_w, Xte_w, vec_w = fs1_tfidf(train_df, test_df)
    Xtr_c, Xte_c, vec_c = fs2_char(train_df, test_df)
    scaler = MinMaxScaler()
    num_tr = scaler.fit_transform(train_df[SOCIAL_COLS].values)
    num_te = scaler.transform(test_df[SOCIAL_COLS].values)
    Xtr = sp.hstack([Xtr_w, Xtr_c, sp.csr_matrix(num_tr)]).tocsr()
    Xte = sp.hstack([Xte_w, Xte_c, sp.csr_matrix(num_te)]).tocsr()
    return Xtr, Xte, (vec_w, vec_c, scaler)

print("Feature builders defined.")

Feature builders defined.


## Cell 5: Feature Experiment Runner

One function for Experiments 7 to 9. Trains, evaluates, saves predictions, and
reports the subgroup breakdown so the fairness effect of each feature set is
visible rather than being hidden behind an aggregate score.

In [6]:
from sklearn.preprocessing import LabelEncoder

def run_feature_experiment(exp_id, feature_name, builder):
    print("="*62)
    print(f"{exp_id.upper()} — {feature_name}")
    print("="*62)

    t0 = time.time()
    X_tr, X_te, _ = builder(tw_train, tw_test)
    print(f"  feature matrix : {X_tr.shape[1]:,} columns")

    model = make_model(BEST_MODEL)

    if NEEDS_ENCODING:
        le = LabelEncoder().fit(y_train)
        model.fit(X_tr, le.transform(y_train))
        y_pred  = le.inverse_transform(model.predict(X_te))
    else:
        model.fit(X_tr, y_train)
        y_pred  = model.predict(X_te)

    y_proba = model.predict_proba(X_te)
    elapsed = time.time() - t0

    res = evaluate_model(y_test, y_pred, y_proba, label=BEST_MODEL)
    res["Experiment No."] = exp_id
    res["Feature Set"]    = feature_name
    res["Model Used"]     = BEST_MODEL
    res["Dataset"]        = "TweetEval"
    res["Error Rate"]     = 1 - res["Accuracy"]
    res["Fit Time (min)"] = round(elapsed/60, 1)

    save_model(model, exp_id, f"{BEST_MODEL}_{exp_id}")
    save_predictions(exp_id, BEST_MODEL, y_test, y_pred, y_proba, sg_test)

    pred_df = load_predictions(exp_id, BEST_MODEL)
    rep = subgroup_report(pred_df)
    for _, r in rep.iterrows():
        res[f"{r['Subgroup']} F1"] = r["Macro F1"]

    print(f"\n  Accuracy   : {res['Accuracy']:.4f}")
    print(f"  Macro F1   : {res['Macro F1']:.4f}")
    print(f"  Error Rate : {res['Error Rate']:.4f}")
    print("\n  Subgroup Macro F1:")
    for _, r in rep.iterrows():
        print(f"    {r['Subgroup']:<14}: {r['Macro F1']:.4f}  (n={int(r['Number of Samples']):,})")
    print()
    return res

feature_results = []

## Cell 6: Reference Row — FS1 Baseline

The Experiment 6 winner rerun on FS1 so that Table 2 has a like-for-like
comparison point. This is the same configuration as Notebook 6, repeated here so
all four rows of Table 2 come from one place.

In [7]:
res = run_feature_experiment("exp06ref", "TF-IDF unigram + bigram (reference)", fs1_tfidf)
feature_results.append(res)

EXP06REF — TF-IDF unigram + bigram (reference)
  feature matrix : 50,000 columns
  saved model -> exp06ref_LogisticRegression_exp06ref.pkl
  saved predictions -> exp06ref_LogisticRegression_TweetEval.parquet  (12,284 rows)

  Accuracy   : 0.5865
  Macro F1   : 0.5453
  Error Rate : 0.4135

  Subgroup Macro F1:
    formal        : 0.5367  (n=10,398)
    other         : 0.5707  (n=1,116)
    emoji-heavy   : 0.5437  (n=696)
    slang-heavy   : 0.6812  (n=60)
    sarcasm       : 0.5702  (n=14)



## Cell 7: Experiment 7 — Character n-grams

Character n-grams of length 3 to 5 with word boundaries. The rationale is that
informal spellings such as 'gr8', 'luv' and 'sooo' share character patterns with
their standard forms even when the whole word never appears in the vocabulary.

If the slang-heavy subgroup improves here while formal stays flat, that supports
the claim that word-level TF-IDF systematically disadvantages informal writing.

In [8]:
res = run_feature_experiment("exp07", "Character n-grams (3-5)", fs2_char)
feature_results.append(res)

EXP07 — Character n-grams (3-5)
  feature matrix : 30,000 columns
  saved model -> exp07_LogisticRegression_exp07.pkl
  saved predictions -> exp07_LogisticRegression_TweetEval.parquet  (12,284 rows)

  Accuracy   : 0.6121
  Macro F1   : 0.5846
  Error Rate : 0.3879

  Subgroup Macro F1:
    formal        : 0.5763  (n=10,398)
    other         : 0.5968  (n=1,116)
    emoji-heavy   : 0.5834  (n=696)
    slang-heavy   : 0.6101  (n=60)
    sarcasm       : 0.6556  (n=14)



## Cell 8: Experiment 8 — Social Media Features

TF-IDF plus three engineered features: emoji density, slang ratio and punctuation
intensity. These are the same quantities used to define the linguistic subgroups.

This is the most direct test of whether making informal-language properties
explicit in the feature space reduces the subgroup fairness gap.

In [9]:
res = run_feature_experiment("exp08", "TF-IDF + social media features", fs3_social)
feature_results.append(res)

EXP08 — TF-IDF + social media features
  feature matrix : 50,003 columns
  saved model -> exp08_LogisticRegression_exp08.pkl
  saved predictions -> exp08_LogisticRegression_TweetEval.parquet  (12,284 rows)

  Accuracy   : 0.5855
  Macro F1   : 0.5439
  Error Rate : 0.4145

  Subgroup Macro F1:
    formal        : 0.5359  (n=10,398)
    other         : 0.5532  (n=1,116)
    emoji-heavy   : 0.5500  (n=696)
    slang-heavy   : 0.6785  (n=60)
    sarcasm       : 0.7500  (n=14)



## Cell 9: Experiment 9 — Hybrid Feature Set

Word TF-IDF, character n-grams and the three engineered features combined.

Maximum coverage, but also the highest dimensionality. Worth watching whether the
gain over Experiment 8 justifies the extra complexity, or whether the character
and engineered features are capturing the same signal twice.

In [10]:
res = run_feature_experiment("exp09", "Hybrid feature set", fs4_hybrid)
feature_results.append(res)

EXP09 — Hybrid feature set
  feature matrix : 80,003 columns
  saved model -> exp09_LogisticRegression_exp09.pkl
  saved predictions -> exp09_LogisticRegression_TweetEval.parquet  (12,284 rows)

  Accuracy   : 0.6165
  Macro F1   : 0.5945
  Error Rate : 0.3835

  Subgroup Macro F1:
    formal        : 0.5881  (n=10,398)
    other         : 0.5844  (n=1,116)
    emoji-heavy   : 0.5963  (n=696)
    slang-heavy   : 0.6834  (n=60)
    sarcasm       : 0.7500  (n=14)



## Cell 10: Table 2 — Feature Engineering Comparison

In [11]:
table2 = pd.DataFrame(feature_results)

cols = ["Experiment No.", "Feature Set", "Model Used", "Dataset",
        "Accuracy", "Macro F1", "Weighted F1", "Error Rate"]
sg_cols = [c for c in table2.columns if c.endswith(" F1") and c not in ("Macro F1", "Weighted F1")]
table2 = table2[cols + sg_cols]

best_idx = table2["Macro F1"].idxmax()
table2["Selected for Final Model?"] = ["YES" if i == best_idx else "No"
                                        for i in table2.index]

print("="*90)
print("TABLE 2 — FEATURE ENGINEERING COMPARISON")
print("="*90)
print(table2.to_string(index=False))

BEST_FEATURE_SET = table2.loc[best_idx, "Feature Set"]
BEST_FEATURE_EXP = table2.loc[best_idx, "Experiment No."]
print(f"\nSelected feature set: {BEST_FEATURE_SET}  ({BEST_FEATURE_EXP})")

print("\nCheck the subgroup columns, not just Macro F1.")
print("A feature set that lifts sarcasm or slang F1 without moving the overall")
print("score is the more interesting result for this study.")

save_result_table(table2, "Table2_Feature_Comparison")

TABLE 2 — FEATURE ENGINEERING COMPARISON
Experiment No.                         Feature Set         Model Used   Dataset  Accuracy  Macro F1  Weighted F1  Error Rate  formal F1  other F1  emoji-heavy F1  slang-heavy F1  sarcasm F1 Selected for Final Model?
      exp06ref TF-IDF unigram + bigram (reference) LogisticRegression TweetEval  0.586535  0.545296     0.562201    0.413465   0.536674  0.570690        0.543689        0.681172    0.570238                        No
         exp07             Character n-grams (3-5) LogisticRegression TweetEval  0.612097  0.584601     0.598870    0.387903   0.576333  0.596785        0.583376        0.610136    0.655556                        No
         exp08      TF-IDF + social media features LogisticRegression TweetEval  0.585477  0.543923     0.561313    0.414523   0.535939  0.553194        0.549978        0.678502    0.750000                        No
         exp09                  Hybrid feature set LogisticRegression TweetEval  0.616493  0.59

PosixPath('/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/results/Table2_Feature_Comparison.csv')

## Cell 11: Experiment 10 — Final Hyperparameter Tuning

Full GridSearchCV on the winning model and feature set. Scoring is macro F1
throughout — using accuracy would reward a model for ignoring the 19% negative
class entirely.

Brier Score is recorded because the AIF360 metrics in Experiment 20 assume
calibrated probability inputs. A Brier Score above 0.25 means those fairness
values need qualifying in the write-up.

In [12]:
BUILDERS = {
    "TF-IDF unigram + bigram (reference)": fs1_tfidf,
    "Character n-grams (3-5)":             fs2_char,
    "TF-IDF + social media features":      fs3_social,
    "Hybrid feature set":                  fs4_hybrid,
}

GRIDS = {
    "LogisticRegression": {"C": [0.01, 0.1, 1, 10, 100], "penalty": ["l2"],
                           "solver": ["liblinear"]},
    "LinearSVM":          {"estimator__C": [0.001, 0.01, 0.1, 1, 10]},
    "MultinomialNB":      {"alpha": [0.01, 0.1, 0.5, 1.0, 2.0]},
    "RandomForest":       {"n_estimators": [200, 400], "max_features": ["sqrt"],
                           "min_samples_leaf": [1, 3, 5]},
    "XGBoost":            {"n_estimators": [200, 400], "learning_rate": [0.05, 0.1],
                           "max_depth": [5, 7]},
}

print("="*62)
print("EXP 10 — HYPERPARAMETER TUNING")
print("="*62)
print(f"  model       : {BEST_MODEL}")
print(f"  feature set : {BEST_FEATURE_SET}\n")

X_tr, X_te, fitted_transformers = BUILDERS[BEST_FEATURE_SET](tw_train, tw_test)

before = evaluate_model(
    y_test,
    load_predictions(BEST_FEATURE_EXP, BEST_MODEL)["y_pred"].values,
    label=BEST_MODEL)

t0 = time.time()
if NEEDS_ENCODING:
    le_final = LabelEncoder().fit(y_train)
    gs = GridSearchCV(make_model(BEST_MODEL), GRIDS[BEST_MODEL],
                      cv=5, scoring="f1_macro", n_jobs=-1)
    gs.fit(X_tr, le_final.transform(y_train))
    y_pred = le_final.inverse_transform(gs.best_estimator_.predict(X_te))
else:
    gs = GridSearchCV(make_model(BEST_MODEL), GRIDS[BEST_MODEL],
                      cv=5, scoring="f1_macro", n_jobs=-1)
    gs.fit(X_tr, y_train)
    y_pred = gs.best_estimator_.predict(X_te)

y_proba = gs.best_estimator_.predict_proba(X_te)
elapsed = time.time() - t0

after = evaluate_model(y_test, y_pred, y_proba, label=BEST_MODEL)

print(f"  best params : {gs.best_params_}")
print(f"  fit time    : {elapsed/60:.1f} min\n")
print(f"  Macro F1 before : {before['Macro F1']:.4f}")
print(f"  Macro F1 after  : {after['Macro F1']:.4f}")
print(f"  improvement     : {after['Macro F1'] - before['Macro F1']:+.4f}")
print(f"  Brier Score     : {after['Brier Score']:.4f}"
      f"{'   <-- above 0.25, qualify fairness values' if after['Brier Score'] > 0.25 else ''}")

save_model(gs.best_estimator_, "exp10", "FinalModel")
save_predictions("exp10", "FinalModel", y_test, y_pred, y_proba, sg_test)

with open(D / "final_transformers.pkl", "wb") as f:
    pickle.dump(fitted_transformers, f)

EXP 10 — HYPERPARAMETER TUNING
  model       : LogisticRegression
  feature set : Hybrid feature set

  best params : {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}
  fit time    : 5.2 min

  Macro F1 before : 0.5945
  Macro F1 after  : 0.5878
  improvement     : -0.0068
  Brier Score     : 0.1822
  saved model -> exp10_FinalModel.pkl
  saved predictions -> exp10_FinalModel_TweetEval.parquet  (12,284 rows)


## Cell 12: Table 3 — Hyperparameter Tuning Results

In [13]:
table3 = pd.DataFrame([{
    "Model":                    BEST_MODEL,
    "Dataset":                  "TweetEval",
    "Feature Set Used":         BEST_FEATURE_SET,
    "Before Tuning Accuracy":   before["Accuracy"],
    "Before Tuning Macro F1":   before["Macro F1"],
    "After Tuning Accuracy":    after["Accuracy"],
    "After Tuning Macro F1":    after["Macro F1"],
    "Best Parameters":          str(gs.best_params_),
    "Macro F1 Improvement":     after["Macro F1"] - before["Macro F1"],
    "Final Rank After Tuning":  1,
    "Brier Score":              after["Brier Score"],
}])

print("="*80)
print("TABLE 3 — HYPERPARAMETER TUNING")
print("="*80)
print(table3.T.to_string())

save_result_table(table3, "Table3_Hyperparameter_Tuning")

with open(D / "final_model_config.json", "w") as f:
    json.dump({"model": BEST_MODEL,
               "feature_set": BEST_FEATURE_SET,
               "feature_exp": BEST_FEATURE_EXP,
               "best_params": {k: str(v) for k, v in gs.best_params_.items()},
               "macro_f1": float(after["Macro F1"]),
               "brier": float(after["Brier Score"])}, f, indent=2)

print("\nNext: Notebook 8 — Experiments 11 to 15 (subgroup evaluation)")

TABLE 3 — HYPERPARAMETER TUNING
                                                                         0
Model                                                   LogisticRegression
Dataset                                                          TweetEval
Feature Set Used                                        Hybrid feature set
Before Tuning Accuracy                                            0.616493
Before Tuning Macro F1                                            0.594538
After Tuning Accuracy                                             0.598095
After Tuning Macro F1                                             0.587758
Best Parameters          {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}
Macro F1 Improvement                                              -0.00678
Final Rank After Tuning                                                  1
Brier Score                                                       0.182236
  saved table -> Table3_Hyperparameter_Tuning.csv

Next: Notebook 8 